# Enterprise E-Commerce International Shipping & Late Delivery Risk Engine

### 1. Strategic Corporate Context
In global logistics and e-commerce fulfillment, missing a delivery deadline directly damages customer retention, slashes lifetime value, and forces expensive customer care interventions. Classifying a high-risk shipment that will experience a late arrival as "on time" (a False Negative) creates severe operational friction. This engine proactively identifies supply chain bottlenecks and high-risk deliveries using real-world operational tracking data.

### 2. Engineering Architecture
To ensure seamless integration into a production-grade microservice or multi-container Docker platform, the technical implementation enforces:
* **Perimeter Gateways:** Strict runtime data type validation via Pydantic to filter out bad payloads before they interact with model memory.
* **Feature Isolation Lines:** Distinct pipelines for scaling continuous metrics and vectorizing categorical properties via an immutable `ColumnTransformer`.
* **Telemetry and Lineage Logs:** An active local SQLite MLflow tracking server recording precise training metrics, tuning variables, and performance reports.

In [1]:
# =====================================================================
# CELL 1: SYSTEM INITIALIZATION & DEPENDENCY SETUP
# =====================================================================
!pip install mlflow pydantic xgboost -q

import mlflow
import pydantic
import sklearn
import pandas as pd
import numpy as np

# Bind tracking to a local relational storage engine to isolate metadata safely
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("ECommerce_Shipping_Logistics_Architecture")

print("--- Logistical Framework Initialized ---")
print(f"MLflow State Engine: Active (SQLite Backend)")
print(f"Pydantic Version: {pydantic.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 936.9/936.9 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2026/06/16 01:20:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/16 01:20:12 INFO mlflow.store.db.utils: Updating database tables
2026/06/16 01:20:16 INFO mlflow.tracking.fluent: Experiment with name 'ECommerce_Shipping_Logistics_Architecture' does not exist. Creating a new experiment.


--- Logistical Framework Initialized ---
MLflow State Engine: Active (SQLite Backend)
Pydantic Version: 2.12.3


### 3. Data Ingestion & Strategic Structural Cleansing
We consume a live production snapshot of retail shipping details (~11,000 application histories). Before partitions hit training matrices, we drop system structural primary keys (`ID`) to prevent the tree-boosting models from optimizing for indexing patterns instead of authentic logistical signal.

In [4]:
# =====================================================================
# CELL 2: DATA INGESTION & PRODUCTION TRAIN/TEST SPLITTING
# =====================================================================
import pandas as pd
from sklearn.model_selection import train_test_split

# Verified evergreen public mirror for the Kaggle International E-Commerce Delivery Dataset
dataset_url = "https://raw.githubusercontent.com/jeeniagogoi1/Ecommerce-Shipping-Prediction/main/Train.csv"
raw_shipping_data = pd.read_csv(dataset_url)

print("--- Raw Logistics Data Structural Audit ---")
print(f"Total Operational Deliveries Segmented: {raw_shipping_data.shape[0]}")
print(f"Total Column Metrics: {raw_shipping_data.shape[1]}")

# DATA HYGIENE: Drop metadata sequence keys to eliminate baseline alignment bias
cleaned_shipping_data = raw_shipping_data.drop(columns=['ID'])

# Target Definition (Reached.on.Time_Y.N: 1 = Late Delivery, 0 = On Time Delivery)
X = cleaned_shipping_data.drop(columns=['Reached.on.Time_Y.N'])
y = cleaned_shipping_data['Reached.on.Time_Y.N']

# Stratify to securely lock down target ratios uniformly across splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Lane Data Geometry: {X_train.shape}")
print(f"Testing Lane Data Geometry: {X_test.shape}")

--- Raw Logistics Data Structural Audit ---
Total Operational Deliveries Segmented: 10999
Total Column Metrics: 12
Training Lane Data Geometry: (8799, 10)
Testing Lane Data Geometry: (2200, 10)


### 4. Data Validation Interface Contract (Pydantic Gateway)
We define our rigorous **Data Gateway Contract** prior to modeling. This contract maps specific acceptable inputs for live prediction calls. If web application payloads fail to match these exact types or exceed realistic physical and financial bounds, the contract rejects the transaction before downstream microservice failures occur.

In [6]:
# =====================================================================
# CELL 3: DATA VALIDATION GATEWAY CONTRACT
# =====================================================================
from pydantic import BaseModel, Field, field_validator
from typing import Literal

class ShippingApplicationSchema(BaseModel):
    Warehouse_block: Literal['A', 'B', 'C', 'D', 'F']
    Mode_of_Shipment: Literal['Ship', 'Flight', 'Road']
    Customer_care_calls: int = Field(..., ge=2, le=10)
    Customer_rating: int = Field(..., ge=1, le=5)
    Cost_of_the_Product: float = Field(..., ge=50.0, le=1000.0)
    Prior_purchases: int = Field(..., ge=2, le=20)
    Product_importance: Literal['low', 'medium', 'high']
    Gender: Literal['F', 'M']
    Discount_offered: float = Field(..., ge=0.0, le=100.0)
    Weight_in_gms: float = Field(..., ge=500.0, le=15000.0)

    @field_validator('Cost_of_the_Product', 'Weight_in_gms')
    @classmethod
    def verify_positive_metrics(cls, value: float) -> float:
        if value <= 0:
            raise ValueError("Physical and financial measurements must be positive non-zero assets.")
        return value

print("Logistics boundary runtime contract safely built.")

Logistics boundary runtime contract safely built.


### 5. Decoupled Multi-Channel Transformation Matrices
Real-world delivery features feature varied arrays of sparse text classifications alongside skewed continuous numeric entries. We construct parallel isolation tracks using a `ColumnTransformer` block to process features without state leaking.

In [7]:
# =====================================================================
# CELL 4: MULTI-CHANNEL ISOLATION PREPROCESSING TRANSFORMERS
# =====================================================================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numerical_logistics_fields = [
    'Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product',
    'Prior_purchases', 'Discount_offered', 'Weight_in_gms'
]
categorical_logistics_fields = [
    'Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender'
]

numerical_processing_route = Pipeline(steps=[
    ('median_imputer', SimpleImputer(strategy='median')),
    ('matrix_scaler', StandardScaler())
])

categorical_processing_route = Pipeline(steps=[
    ('constant_imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('one_hot_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
])

production_preprocessing_gate = ColumnTransformer(transformers=[
    ('num_track', numerical_processing_route, numerical_logistics_fields),
    ('cat_track', categorical_processing_route, categorical_logistics_fields)
])

print("Data distribution lanes securely clamped.")

Data distribution lanes securely clamped.


### 6. Instrumented Gradient Boosting & Logistical Performance Auditing
We deploy an optimized tree-boosting engine (`XGBClassifier`) fully instrumented through an open MLflow auditing block. It automatically tracks hyperparameter sets, pipeline steps, and custom structural performance dimensions directly within our localized SQLite tracking repository.

In [8]:
# =====================================================================
# CELL 5: INSTRUMENTED TRAINING LOOP & PERFORMANCE AUDITING
# =====================================================================
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import mlflow.sklearn

# Dynamically calculate target class balance weights
imbalance_coefficient = float(y_train.value_counts()[0] / y_train.value_counts()[1])

shipping_risk_pipeline = Pipeline(steps=[
    ('preprocessing_gate', production_preprocessing_gate),
    ('xgboost_engine', XGBClassifier(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=imbalance_coefficient,
        random_state=42,
        n_jobs=-1
    ))
])

mlflow.sklearn.autolog(log_models=True)

print("Opening MLflow channel. Launching optimization cluster...")
with mlflow.start_run(run_name="Logistics_Late_Prediction_Production") as active_run:

    shipping_risk_pipeline.fit(X_train, y_train)

    predictions = shipping_risk_pipeline.predict(X_test)
    probabilities = shipping_risk_pipeline.predict_proba(X_test)[:, 1]

    logistics_auc = roc_auc_score(y_test, probabilities)
    mlflow.log_metric("operational_test_auc", logistics_auc)

    print("\n=====================================================================")
    print("--- METRICS TRACE RECORDED ---")
    print(f"Logged Active Run ID: {active_run.info.run_id}")
    print(f"Target Performance AUC Score: {logistics_auc:.4f}")
    print("=====================================================================\n")
    print(classification_report(y_test, predictions))

Opening MLflow channel. Launching optimization cluster...


2026/06/16 01:23:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/16 01:23:20 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils


--- METRICS TRACE RECORDED ---
Logged Active Run ID: 9445de7aaa1c4f47b544f08164704ca5
Target Performance AUC Score: 0.7375

              precision    recall  f1-score   support

           0       0.56      0.97      0.71       887
           1       0.96      0.48      0.64      1313

    accuracy                           0.68      2200
   macro avg       0.76      0.73      0.68      2200
weighted avg       0.80      0.68      0.67      2200



### 7. Hard-Drive Asset Serialization
We export the verified end-to-end processing and decision graph directly as a unified portable `.pkl` artifact, preparing our application stack for integration into external web layer engines.

In [9]:
# =====================================================================
# CELL 6: DIRECT CORE SERIALIZATION & PIPELINE EXPORT
# =====================================================================
from google.colab import files
import joblib

joblib.dump(shipping_risk_pipeline, 'shipping_risk_pipeline.pkl')
files.download('shipping_risk_pipeline.pkl')
print("Successfully generated and downloaded 'shipping_risk_pipeline.pkl'!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully generated and downloaded 'shipping_risk_pipeline.pkl'!
